In [ ]:
#drive 마운트
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# YOLOv8 설치
!pip install ultralytics --upgrade

# EasyOCR 및 OpenCV
!pip install easyocr
!pip install opencv-python-headless


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 963.8/963.8 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 292.1/292.1 kB 25.7 MB/s eta 0:00:00


# json -> YOLO 포맷 변환




In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import drive
import os, json, random, shutil


# ✅ 2. 경로 설정
json_dir = '/content/drive/MyDrive/car_license/dataset/TL_차량번호판인식_교차로_[cr14]동안사거리_05번'
img_dir = '/content/drive/MyDrive/car_license/dataset/TS_차량번호판인식_교차로_[cr14]동안사거리_05번'
output_base = '/content/drive/MyDrive/car_license/processed_dataset'

# ✅ 3. YOLO 형식 변환 함수
def convert_bbox(bbox, img_width, img_height):
    x, y, w, h = bbox
    x_c = (x + w / 2) / img_width
    y_c = (y + h / 2) / img_height
    w /= img_width
    h /= img_height
    return x_c, y_c, w, h

def convert_single_json(json_path, output_txt_path, img_width=1920, img_height=1080):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    annotations = data['Learning_Data_Info']['annotations'][0]['license_plate']
    lines = []
    for obj in annotations:
        if obj.get('class_ID') == 'unknown':
            continue
        bbox = obj['bbox']
        x_c, y_c, w_n, h_n = convert_bbox(bbox, img_width, img_height)
        lines.append(f"0 {x_c:.6f} {y_c:.6f} {w_n:.6f} {h_n:.6f}")

    os.makedirs(os.path.dirname(output_txt_path), exist_ok=True)
    with open(output_txt_path, 'w') as f:
        f.write("\n".join(lines))

# ✅ 4. 이미지 + JSON 파일 쌍으로 정리
img_files = sorted([f for f in os.listdir(img_dir) if f.lower().endswith('.jpg')])
json_files = sorted([f for f in os.listdir(json_dir) if f.lower().endswith('.json')])

dataset = []
for img_file in img_files:
    base = os.path.splitext(img_file)[0]
    json_file = base + '.json'
    if json_file in json_files:
        dataset.append((img_file, json_file))

print(f"✅ 유효한 데이터 개수: {len(dataset)}")

# ✅ 5. train/val 분할 (80/20)
random.shuffle(dataset)
split_index = int(len(dataset) * 0.8)
train_set = dataset[:split_index]
val_set = dataset[split_index:]

# ✅ 6. 파일 복사 및 라벨 변환
for split_name, split_data in [('train', train_set), ('val', val_set)]:
    img_out_dir = os.path.join(output_base, 'images', split_name)
    lbl_out_dir = os.path.join(output_base, 'labels', split_name)
    os.makedirs(img_out_dir, exist_ok=True)
    os.makedirs(lbl_out_dir, exist_ok=True)

    for img_file, json_file in split_data:
        base = os.path.splitext(img_file)[0]

        # 이미지 복사
        shutil.copy(os.path.join(img_dir, img_file), os.path.join(img_out_dir, img_file))

        # 라벨 변환 및 저장
        convert_single_json(
            json_path=os.path.join(json_dir, json_file),
            output_txt_path=os.path.join(lbl_out_dir, base + '.txt'),
            img_width=1920,
            img_height=1080
        )

print("✅ 이미지 및 라벨 변환 + train/val 분할 완료!")


✅ 유효한 데이터 개수: 915
✅ 이미지 및 라벨 변환 + train/val 분할 완료!


# yaml 파일 생성


In [ ]:
yaml_content = """
train: /content/drive/MyDrive/car_license/processed_dataset/images/train
val: /content/drive/MyDrive/car_license/processed_dataset/images/val
test: /content/drive/MyDrive/car_license/processed_dataset/images/test

nc: 1
names: ['license_plate']
"""

with open("data.yaml", "w") as f:
    f.write(yaml_content)

# 결과 확인
!cat data.yaml




train: /content/drive/MyDrive/car_license/processed_dataset/images/train
val: /content/drive/MyDrive/car_license/processed_dataset/images/val
test: /content/drive/MyDrive/car_license/processed_dataset/images/test

nc: 1
names: ['license_plate']


### 욜로8 모델링 ###


In [ ]:
from ultralytics import YOLO
# 사전학습된 YOLO 모델 불러오기
model = YOLO("yolov8n.pt")  # 또는 yolov8s.pt 등

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
# 하이퍼파라미터 튜닝 (진화 기반)
model.train(
    data="/content/drive/MyDrive/car_license/processed_dataset/data.yaml",   # 데이터셋 설정 파일
    imgsz=640,                  # 입력 이미지 크기
    epochs=20,                  # 각 조합 실험의 에폭 수 (짧게 설정)
    batch=16,                   # 배치 크기
    name="lp_evolve",           # 실험 이름
    project="runs/evolve",      # 저장 경로
)


Ultralytics 8.3.174 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/car_license/processed_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=lp_evolve, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=

Overriding model.yaml nc=80 with nc=1

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics

 16                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
 17            [-1, 12]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 18                  -1  1    123648  ultralytics.nn.modules.block.C2f             [192, 128, 1]                 
 19                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
 20             [-1, 9]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 21                  -1  1    493056  ultralytics.nn.modules.block.C2f             [384, 256, 1]                 
 22        [15, 18, 21]  1    751507  ultralytics.nn.modules.head.Detect           [1, [64, 128, 256]]           
Model summary: 129 layers, 3,011,043 parameters, 3,011,027 gradients, 8.2 GFLOPs

Transferred 319/355 items from pretrained weights
Freezing layer 'model.22.dfl.conv.weight'
AMP: running Automatic Mixed

AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.4±0.1 ms, read: 1.1±0.4 MB/s, size: 460.6 KB)


train: Scanning /content/drive/MyDrive/car_license/processed_dataset/labels/train.cache... 732 images, 0 backgrounds, 0 corrupt: 100%|██████████| 732/732 [00:00<?, ?it/s]


...

                   all        183        215      0.932      0.958      0.985      0.735



20 epochs completed in 0.056 hours.
Optimizer stripped from runs/evolve/lp_evolve/weights/last.pt, 6.2MB
Optimizer stripped from runs/evolve/lp_evolve/weights/best.pt, 6.2MB

Validating runs/evolve/lp_evolve/weights/best.pt...
Ultralytics 8.3.174 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 6/6 [00:01<00:00,  3.79it/s]


                   all        183        215      0.936       0.96      0.985      0.742
Speed: 0.3ms preprocess, 1.8ms inference, 0.0ms loss, 2.4ms postprocess per image
Results saved to runs/evolve/lp_evolve


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f6ba85091d0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

### best.pt 모델 드라이브에 저장 ###


In [ ]:
import shutil

# 원본 가중치 경로 (YOLOv8 훈련 결과)
src = "/content/runs/evolve/lp_evolve/weights/best.pt"

# 드라이브 저장 경로
dst = "/content/drive/MyDrive/car_license"

# 파일 복사
shutil.copy(src, dst)
print("✅ best.pt 모델 가중치를 드라이브에 저장했습니다!")


✅ best.pt 모델 가중치를 드라이브에 저장했습니다!


###결과 예시 이미지 시각화(새로운 이미지 넣어서) ###


In [ ]:
!ls /content/drive/MyDrive/car_license/processed_dataset/images/


train  val


In [ ]:
!mkdir -p /content/drive/MyDrive/car_license/processed_dataset/images/test


In [ ]:
model = YOLO("runs/evolve/lp_evolve/weights/best.pt")

# 모든 이미지 경로 리스트
import glob

test_images = glob.glob(test_img_dir + "/*.jpg")  # 확장자는 필요에 따라 수정 (png 등)

# 예측 수행 (save=True 하면 runs/detect/predict에 저장됨)
results = model.predict(
    source="/content/drive/MyDrive/car_license/processed_dataset/images/test",  # ← 여기 경로를 본인 경로로 수정
    conf=0.1,   # 낮춰서 더 많은 탐지 시도
)

NameError: name 'YOLO' is not defined

In [ ]:
import os
from IPython.display import Image, display

# 결과 저장된 폴더
result_dir = "runs/detect/predict"

# 이미지 파일 목록
result_images = [f for f in os.listdir(result_dir) if f.endswith(('.jpg', '.png'))]

# 이미지 시각화
for img_file in result_images[:15]:
    display(Image(filename=os.path.join(result_dir, img_file)))


Output hidden; open in https://colab.research.google.com to view.

### 번호판 영역 Crop ###



In [ ]:
from ultralytics import YOLO

# 학습한 모델 로드
model = YOLO("/content/drive/MyDrive/car_license/models/yolov11_best (1).pt")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
results = model.predict(
    source='/content/drive/MyDrive/car_license/추출이미지폴더',
    conf=0.3,
    save_crop=True,
    save=True,
    project='/content/drive/MyDrive/car_license/',  # 결과 저장 최상위 경로를 드라이브로 지정
    name='cropimagedata'  # 결과 폴더 이름 지정
)



image 1/100 /content/drive/MyDrive/car_license/추출이미지폴더/-ZZ_1_F1_100_100_N_241207-EventVideo-KCity-FallingObject-100m-Night-60-_Front_Pickup_frame_00029_jpg.rf.2c9b5d6310cb393e15e4e9733d889cb9.jpg: 384x640 2 cars, 546.4ms
image 2/100 /content/drive/MyDrive/car_license/추출이미지폴더/-ZZ_1_F1_100_100_N_241207-EventVideo-KCity-FallingObject-100m-Night-60-_Front_Pickup_frame_00065_jpg.rf.571ceed28f1015361fabbc4175bb9f76.jpg: 384x640 2 cars, 477.2ms
image 3/100 /content/drive/MyDrive/car_license/추출이미지폴더/-ZZ_1_F1_100_44_D_241207-EventVideo-KCity-FallingObject-100m-Day-40-_Front_frame_00002_jpg.rf.d0c36ccd193ddaa677d08c46407c8376.jpg: 384x640 1 car, 352.5ms
image 4/100 /content/drive/MyDrive/car_license/추출이미지폴더/-ZZ_1_F1_100_60_D_241207-EventVideo-KCity-FallingObject-100m-Day-50-_Front_Pickup_frame_00012_jpg.rf.0c5684c81ee2fe6665444927a3f973bd.jpg: 384x640 1 car, 366.2ms
image 5/100 /content/drive/MyDrive/car_license/추출이미지폴더/-ZZ_1_F1_100_60_D_241207-Event

In [ ]:
!apt-get update
!apt-get install -y tesseract-ocr
!pip install pytesseract


Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading

In [ ]:
#한글언어팩 설치
!apt-get update
!apt-get install -y tesseract-ocr tesseract-ocr-kor


Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading

In [ ]:
import pytesseract
from PIL import Image
import os

# crop된 번호판 경로
crop_dir = "/content/drive/MyDrive/car_license/runs_detect/crops/license_plate"

ocr_results = []


for fname in sorted(os.listdir(crop_dir)):
    if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
        img_path = os.path.join(crop_dir, fname)
        image = Image.open(img_path)

        # 한글 언어팩 사용, whitelist 적용, --psm 7
        custom_config = r'--psm 7 -c tessedit_char_whitelist=0123456789가나다라마바사아자차카타파하'
        text = pytesseract.image_to_string(image, lang='kor', config=custom_config)

        ocr_results.append((fname, text.strip()))

# 결과 출력
for fname, text in ocr_results:
    print(f"{fname}: '{text}'")

##Easy OCR로 번호판 인식 전 세팅

In [ ]:
!pip install easyocr

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
tesseract-ocr-kor is already the newest version (1:4.00~git30-7274cfa-1.1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.


In [ ]:
# 한글 세팅

# 1. 나눔고딕 폰트 설치
!apt-get -qq update
!apt-get -qq install -y fonts-nanum

# 2. 런타임에 폰트 캐시 갱신
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import matplotlib as mpl

font_dirs = ['/usr/share/fonts/truetype/nanum']
font_files = fm.findSystemFonts(fontpaths=font_dirs)

for font_file in font_files:
    fm.fontManager.addfont(font_file)

# 3. 폰트 이름 확인 후 설정
nanum_font_name = fm.FontProperties(fname=font_files[0]).get_name()
print(f"설정된 폰트 이름: {nanum_font_name}")

# matplotlib에 폰트 반영
mpl.rc('font', family=nanum_font_name)
mpl.rcParams['axes.unicode_minus'] = False


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package fonts-nanum.
(Reading database ... 126284 files and directories currently installed.)
Preparing to unpack .../fonts-nanum_20200506-1_all.deb ...
Unpacking fonts-nanum (20200506-1) ...
Setting up fonts-nanum (20200506-1) ...
Processing triggers for fontconfig (2.13.1-4.2ubuntu5) ...
설정된 폰트 이름: NanumGothicCoding


##yolo로 번호판 객체 탐지 +OCR 인식- 전체 이미지 바운딩 박스 위 인식된 문자열 시각화

In [ ]:
# ✅ 코랩 폰트 설치
!apt-get install -qq fonts-nanum
import matplotlib.pyplot as plt
from PIL import ImageFont, ImageDraw, Image
import numpy as np
import os
import cv2
import re
import easyocr
from ultralytics import YOLO

# ✅ YOLO 모델 로드
model = YOLO("/content/drive/MyDrive/yolov11_best (1) (1).pt")

# ✅ EasyOCR 리더
reader = easyocr.Reader(['ko', 'en'])

# ✅ 허용된 한글 문자만 포함하는 패턴
ALLOWED_HANGUL = '바사아자허하호배가나다라마거너더러머버서어저고노도로모보소오조구누두루무부수우주'
FRONT_PATTERN = re.compile(rf'\d{{2,3}}[{ALLOWED_HANGUL}]')

# 🔧 텍스트 클리닝
def clean_text(text):
    return re.sub(r'[^0-9가-힣]', '', text)

# ✅ 번호판 앞부분 추출
def extract_front_number(text):
    cleaned = clean_text(text)
    match = FRONT_PATTERN.search(cleaned)
    return match.group() if match else ''

# 🔧 OCR 전처리
def preprocess_image_opencv(img):
    if isinstance(img, str):
        image = cv2.imread(img)
    else:
        image = img.copy()

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)
    blurred = cv2.GaussianBlur(gray, (3, 3), 0)
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    resized = cv2.resize(thresh, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)

    return resized

# ✅ 입력 & 출력 폴더
input_folder = "/content/drive/MyDrive/car_license/processed_newnewdataset/images/train"
output_folder = "/content/drive/MyDrive/car_license/yolo_ocr_result_최종"
os.makedirs(output_folder, exist_ok=True)

# 📌 YOLO 클래스 이름 확인
names = model.names
plate_class_id = None
car_class_id = None
for k, v in names.items():
    if v.lower() in ["car_license", "license_plate", "plate"]:
        plate_class_id = k
    if v.lower() in ["car", "vehicle"]:
        car_class_id = k
if plate_class_id is None:
    raise ValueError("❌ 번호판 클래스(car_license) 없음.")
if car_class_id is None:
    print("⚠️ 자동차 클래스(car) 없음. 번호판만 인식됩니다.")

recognized_count = 0  # 인식된 이미지 개수

# ✅ 한글 폰트 설정
font_path = "/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf"
font = ImageFont.truetype(font_path, 32)

# ✅ 이미지 순회
for fname in sorted(os.listdir(input_folder)):
    if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue

    img_path = os.path.join(input_folder, fname)
    print(f"🔍 처리 중: {fname}")

    image = cv2.imread(img_path)
    if image is None:
        print(f"❌ 이미지 로드 실패: {fname}")
        continue

    results = model.predict(source=img_path, verbose=False)
    ocr_found = False  # 해당 이미지에서 OCR 인식 여부

    for r in results:
        boxes = r.boxes.xyxy.cpu().numpy().astype(int)
        cls_ids = r.boxes.cls.cpu().numpy().astype(int)

        for box, cls_id in zip(boxes, cls_ids):
            x1, y1, x2, y2 = box

            # 자동차 박스 그리기
            if cls_id == car_class_id:
                cv2.rectangle(image, (x1, y1), (x2, y2), (255, 255, 0), 2)
                cv2.putText(image, "Car", (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 0), 2, cv2.LINE_AA)
                continue

            # 번호판 처리
            if cls_id == plate_class_id:
                plate_img = image[y1:y2, x1:x2]
                if plate_img.size == 0:
                    continue

                processed_img = preprocess_image_opencv(plate_img)
                easy_text_list = reader.readtext(processed_img, detail=0)
                easy_text = ''.join(easy_text_list) if easy_text_list else ''
                final_text = extract_front_number(easy_text)

                if final_text.strip():  # 인식된 경우만 표시
                    ocr_found = True
                    cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)

                    # ---- 한글 깨짐 방지: PIL로 텍스트 그리기 ----
                    image_pil = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
                    draw = ImageDraw.Draw(image_pil)
                    draw.text((x1, y1 - 40), final_text, font=font, fill=(0, 255, 0))
                    image = cv2.cvtColor(np.array(image_pil), cv2.COLOR_RGB2BGR)

                    print(f"   ✅ OCR 결과: {final_text}")

    # OCR이 하나라도 성공한 경우에만 저장
    if ocr_found:
        save_path = os.path.join(output_folder, fname)
        cv2.imwrite(save_path, image)
        recognized_count += 1
        print(f"💾 저장 완료: {save_path}")
    else:
        print(f"⚠️ OCR 인식 실패: {fname}")
print(f"\n📌 모든 이미지 처리 완료 — 총 {recognized_count}개 이미지 저장됨")

## 크롭된 이미지+ ESRGAN 전처리 이미지 OCR -크롭된 이미지와 결과값만 나옴

In [ ]:
import cv2
from PIL import Image
import easyocr
import os
import re
import numpy as np
import matplotlib.pyplot as plt

# EasyOCR 리더
reader = easyocr.Reader(['ko', 'en'])

# 이미지 폴더 경로
crop_dir = "/content/drive/MyDrive/car_license/크롭된 데이터/ESRGAN"

ocr_results = []

# ✅ 허용된 한글 문자만 포함하는 패턴
ALLOWED_HANGUL = '바사아자허하호배가나다라마거너더러머버서어저고노도로모보소오조구누두루무부수우주'
FRONT_PATTERN = re.compile(rf'\d{{2,3}}[{ALLOWED_HANGUL}]')

# 🔧 텍스트 클리닝 함수
def clean_text(text):
    return re.sub(r'[^0-9가-힣]', '', text)

# ✅ 번호판 앞부분 추출 함수
def extract_front_number(text):
    cleaned = clean_text(text)
    match = FRONT_PATTERN.search(cleaned)
    return match.group() if match else ''

# 🔧 전처리 함수
def preprocess_image_opencv(image_path):
    image = cv2.imread(image_path)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # 대비 향상 (CLAHE)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)

    # 노이즈 제거
    blurred = cv2.GaussianBlur(gray, (3, 3), 0)

    # 이진화
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # 확대
    resized = cv2.resize(thresh, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)

    return resized

# 이미지 순회
for fname in sorted(os.listdir(crop_dir)):
    if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
        img_path = os.path.join(crop_dir, fname)

        # 🔄 전처리 적용
        processed_img = preprocess_image_opencv(img_path)

        # ✅ EasyOCR 수행
        easy_text_list = reader.readtext(processed_img, detail=0)
        easy_text = ''.join(easy_text_list) if easy_text_list else ''

        # ✅ 번호판 앞부분 추출
        final_text = extract_front_number(easy_text)

        ocr_results.append((fname, final_text))

# 결과 필터링
valid_results = [(fname, text) for fname, text in ocr_results if text.strip()]

print("\n✅ 인식된 파일 리스트 및 결과:")
for fname, text in valid_results:
    print(f"{fname}: '{text}'")

# 이미지 표시
print("\n🖼️ 인식된 이미지 표시:")
for fname, text in valid_results:
    img_path = os.path.join(crop_dir, fname)
    img = Image.open(img_path)

    plt.figure()
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"{fname}: '{text}'")
    plt.show()